In [16]:
from pathlib import Path
import time

import pandas as pd
import numpy as np
import joblib
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

PROJECT_ROOT = Path(r"C:\Case Study Trees\AdultCensusStudy")

SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
STUDY_DIR = PROJECT_ROOT / "studies"
RESULT_DIR = PROJECT_ROOT / "results"

STUDY_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
THREADS = 4
BUDGET_SECONDS = 10 * 60

In [17]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
valid_df = pd.read_csv(SPLIT_DIR / "valid.csv")

y_train = train_df["income"].copy()
y_valid = valid_df["income"].copy()

columns_to_drop = ["income", "ROW_ID", "education", "fnlwgt"]

X_train_raw = train_df.drop(columns=columns_to_drop).copy()
X_valid_raw = valid_df.drop(columns=columns_to_drop).copy()

onehot_preprocessor = joblib.load(
    ARTIFACT_DIR / "adult_onehot_preprocessor.joblib"
)

X_train_onehot = onehot_preprocessor.transform(
    X_train_raw
).astype("float32")

X_valid_onehot = onehot_preprocessor.transform(
    X_valid_raw
).astype("float32")

assert len(X_train_onehot) == len(y_train)
assert len(X_valid_onehot) == len(y_valid)
assert X_train_onehot.shape[1] == X_valid_onehot.shape[1]

print("Training:", X_train_onehot.shape)
print("Validation:", X_valid_onehot.shape)

Training: (22775, 90)
Validation: (4881, 90)


## RANDOM FOREST

In [18]:
def rf_objective(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 100, 500, step=100
        ),
        max_depth=trial.suggest_int("max_depth", 8, 30),
        min_samples_split=trial.suggest_int(
            "min_samples_split", 2, 20
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf", 1, 20
        ),
        max_features=trial.suggest_categorical(
            "max_features", ["sqrt", 0.3, 0.6]
        ),
        class_weight=trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        ),
        random_state=SEED,
        n_jobs=THREADS
    )

    model.fit(X_train_onehot, y_train)

    probability = model.predict_proba(X_valid_onehot)[:, 1]

    return roc_auc_score(y_valid, probability)


database_path = (STUDY_DIR / "adult_optuna.db").resolve()

rf_study = optuna.create_study(
    study_name="adult_rf_onehot_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

start_time = time.perf_counter()

rf_study.optimize(
    rf_objective,
    timeout=BUDGET_SECONDS,
    n_jobs=1,
    gc_after_trial=True
)

rf_tuning_seconds = time.perf_counter() - start_time

rf_study.trials_dataframe().to_csv(
    RESULT_DIR / "rf_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", rf_study.best_value)
print("Best parameters:", rf_study.best_params)
print("Tuning seconds:", round(rf_tuning_seconds, 2))

[I 2026-10-16 16:44:40,677] A new study created in RDB with name: adult_rf_onehot_v1
[I 2026-10-16 16:44:42,410] Trial 0 finished with value: 0.9128752513151008 and parameters: {'n_estimators': 200, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 0 with value: 0.9128752513151008.
[I 2026-10-16 16:44:45,423] Trial 1 finished with value: 0.8989185417756845 and parameters: {'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 17, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9128752513151008.
[I 2026-10-16 16:44:52,389] Trial 2 finished with value: 0.9162297915117464 and parameters: {'n_estimators': 300, 'max_depth': 14, 'min_samples_split': 13, 'min_samples_leaf': 3, 'max_features': 0.6, 'class_weight': None}. Best is trial 2 with value: 0.9162297915117464.
[I 2026-10-16 16:44:59,381] Trial 3 finished with value: 0.914910215098185 and parameters

Best validation ROC-AUC: 0.918839222598621
Best parameters: {'n_estimators': 200, 'max_depth': 16, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 0.3, 'class_weight': 'balanced'}
Tuning seconds: 605.18


In [19]:
rf_tuned = RandomForestClassifier(
    **rf_study.best_params,
    random_state=SEED,
    n_jobs=THREADS
)

start_time = time.perf_counter()

rf_tuned.fit(X_train_onehot, y_train)

rf_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = rf_tuned.predict_proba(X_valid_onehot)[:, 1]

rf_prediction_seconds = time.perf_counter() - start_time

rf_results = {
    "Model": "Random Forest",
    "Representation": "One-hot",
    "Version": "Tuned",
    "Validation_ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Validation_AP": average_precision_score(
        y_valid, valid_probability
    ),
    "Validation_Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": rf_fit_seconds,
    "Prediction_Seconds": rf_prediction_seconds,
    "Tuning_Seconds": rf_tuning_seconds
}

display(pd.DataFrame([rf_results]).round(4))

joblib.dump(
    rf_tuned,
    ARTIFACT_DIR / "adult_rf_tuned.joblib"
)

pd.DataFrame([rf_results]).to_csv(
    RESULT_DIR / "rf_tuned_validation.csv",
    index=False
)

pd.Series(rf_study.best_params, name="Value").to_csv(
    RESULT_DIR / "rf_best_parameters.csv",
    index_label="Parameter"
)

print("Tuned Random Forest saved.")

,Model,Representation,Version,Validation_ROC_AUC,Validation_AP,Validation_Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,Random Forest,One-hot,Tuned,0.9188,0.8042,0.3482,2.7493,0.092,605.1804


Tuned Random Forest saved.


## XGBOOST

In [25]:
from xgboost import XGBClassifier

native_preprocessor = joblib.load(
    ARTIFACT_DIR / "adult_native_preprocessor.joblib"
)

feature_names = native_preprocessor.get_feature_names_out()

X_train_xgb = pd.DataFrame(
    native_preprocessor.transform(X_train_raw),
    columns=feature_names,
    index=X_train_raw.index
)

X_valid_xgb = pd.DataFrame(
    native_preprocessor.transform(X_valid_raw),
    columns=feature_names,
    index=X_valid_raw.index
)

numeric_columns = [
    column for column in feature_names
    if column.startswith("numeric__")
]

cat_columns = [
    column for column in feature_names
    if column.startswith("categorical__")
]

print("Categorical features:")
print(cat_columns)

for data in [X_train_xgb, X_valid_xgb]:
    data[numeric_columns] = data[numeric_columns].astype("float32")

# Use training-derived category definitions for both splits.
category_levels = {}

for column in cat_columns:
    train_values = X_train_xgb[column].astype(str)
    valid_values = X_valid_xgb[column].astype(str)

    levels = sorted(train_values.unique().tolist())

    if "Missing" not in levels:
        levels.append("Missing")

    category_levels[column] = levels

    valid_values = valid_values.where(
        valid_values.isin(levels),
        "Missing"
    )

    X_train_xgb[column] = pd.Categorical(
        train_values, categories=levels
    )

    X_valid_xgb[column] = pd.Categorical(
        valid_values, categories=levels
    )

assert X_train_xgb.columns.equals(X_valid_xgb.columns)
assert len(X_train_xgb) == len(y_train)
assert len(X_valid_xgb) == len(y_valid)

joblib.dump(
    {
        "original_columns": feature_names.tolist(),
        "numeric_columns": numeric_columns,
        "categorical_columns": cat_columns,
        "category_levels": category_levels,
        "model_columns": feature_names.tolist()
    },
    ARTIFACT_DIR / "adult_xgb_input_info.joblib"
)


def xgb_objective(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        max_depth=trial.suggest_int("max_depth", 3, 9),
        min_child_weight=trial.suggest_float(
            "min_child_weight", 1, 30, log=True
        ),
        subsample=trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        colsample_bytree=trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda", 0.1, 20, log=True
        ),
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        random_state=SEED,
        n_jobs=THREADS
    )

    model.fit(X_train_xgb, y_train)

    probability = model.predict_proba(X_valid_xgb)[:, 1]

    return roc_auc_score(y_valid, probability)


database_path = (STUDY_DIR / "adult_optuna.db").resolve()

xgb_study = optuna.create_study(
    study_name="adult_xgb_native_10min_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

start_time = time.perf_counter()

xgb_study.optimize(
    xgb_objective,
    timeout=10 * 60,
    n_jobs=1,
    gc_after_trial=True
)

xgb_tuning_seconds = time.perf_counter() - start_time

xgb_study.trials_dataframe().to_csv(
    RESULT_DIR / "xgb_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", xgb_study.best_value)
print("Best parameters:", xgb_study.best_params)
print("Tuning seconds:", round(xgb_tuning_seconds, 2))

[I 2026-10-16 17:53:05,491] A new study created in RDB with name: adult_xgb_native_10min_v1


Categorical features:
['categorical__workclass', 'categorical__marital.status', 'categorical__occupation', 'categorical__relationship', 'categorical__race', 'categorical__sex', 'categorical__native.country']


[I 2026-10-16 17:53:07,511] Trial 0 finished with value: 0.9206391665978133 and parameters: {'n_estimators': 500, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 7.6611007077713635, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 9.842315738502599}. Best is trial 0 with value: 0.9206391665978133.
[I 2026-10-16 17:53:08,863] Trial 1 finished with value: 0.9248387681658359 and parameters: {'n_estimators': 700, 'learning_rate': 0.06803900745073706, 'max_depth': 3, 'min_child_weight': 27.08160864249967, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.26425260575499165}. Best is trial 1 with value: 0.9248387681658359.
[I 2026-10-16 17:53:10,173] Trial 2 finished with value: 0.9291555353585428 and parameters: {'n_estimators': 400, 'learning_rate': 0.04141536110538596, 'max_depth': 6, 'min_child_weight': 2.69265525148

Best validation ROC-AUC: 0.9324586649774619
Best parameters: {'n_estimators': 700, 'learning_rate': 0.12820666962637578, 'max_depth': 3, 'min_child_weight': 4.4172881699117985, 'subsample': 0.9966564582195031, 'colsample_bytree': 0.7706234746224288, 'reg_alpha': 0.001990043285936671, 'reg_lambda': 0.429959734228568}
Tuning seconds: 601.71


In [26]:
xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    random_state=SEED,
    n_jobs=THREADS
)

start_time = time.perf_counter()

xgb_tuned.fit(X_train_xgb, y_train)

xgb_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = xgb_tuned.predict_proba(
    X_valid_xgb
)[:, 1]

xgb_prediction_seconds = time.perf_counter() - start_time

xgb_results = {
    "Model": "XGBoost",
    "Representation": "Native",
    "Version": "Tuned",
    "Study_Name": xgb_study.study_name,
    "Budget_Seconds": 600,
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, valid_probability
    ),
    "Validation_AP": average_precision_score(
        y_valid, valid_probability
    ),
    "Validation_Log_Loss": log_loss(
        y_valid, valid_probability
    ),
    "Fit_Seconds": xgb_fit_seconds,
    "Prediction_Seconds": xgb_prediction_seconds,
    "Tuning_Seconds": xgb_tuning_seconds
}

display(pd.DataFrame([xgb_results]).round(4))

joblib.dump(
    xgb_tuned,
    ARTIFACT_DIR / "adult_xgb_tuned.joblib"
)

xgb_tuned.save_model(
    ARTIFACT_DIR / "adult_xgb_tuned.ubj"
)

pd.DataFrame([xgb_results]).to_csv(
    RESULT_DIR / "xgb_tuned_validation.csv",
    index=False
)

pd.Series(xgb_study.best_params, name="Value").to_csv(
    RESULT_DIR / "xgb_best_parameters.csv",
    index_label="Parameter"
)

print("Tuned XGBoost and input metadata saved.")

,Model,Representation,Version,Study_Name,Budget_Seconds,Validation_ROC_AUC,Validation_AP,Validation_Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,XGBoost,Native,Tuned,adult_xgb_native_10min_v1,600,0.9325,0.834,0.2702,1.1946,0.0302,601.7067


Tuned XGBoost and input metadata saved.


## LIGHT GBM

In [20]:
from lightgbm import LGBMClassifier


native_preprocessor = joblib.load(
    ARTIFACT_DIR / "adult_native_preprocessor.joblib"
)

feature_names = native_preprocessor.get_feature_names_out()

X_train_native = pd.DataFrame(
    native_preprocessor.transform(X_train_raw),
    columns=feature_names,
    index=X_train_raw.index
)

X_valid_native = pd.DataFrame(
    native_preprocessor.transform(X_valid_raw),
    columns=feature_names,
    index=X_valid_raw.index
)

numeric_columns = [
    column for column in feature_names
    if column.startswith("numeric__")
]

cat_columns = [
    column for column in feature_names
    if column.startswith("categorical__")
]

print("Categorical features:")
print(cat_columns)

for data in [X_train_native, X_valid_native]:
    data[numeric_columns] = data[numeric_columns].astype("float32")

category_levels = {}

for column in cat_columns:
    train_values = X_train_native[column].astype(str)
    valid_values = X_valid_native[column].astype(str)

    levels = sorted(train_values.unique().tolist())

    if "Missing" not in levels:
        levels.append("Missing")

    category_levels[column] = levels

    valid_values = valid_values.where(
        valid_values.isin(levels), "Missing"
    )

    X_train_native[column] = pd.Categorical(
        train_values, categories=levels
    )

    X_valid_native[column] = pd.Categorical(
        valid_values, categories=levels
    )

safe_names = [
    f"feature_{i}" for i in range(len(feature_names))
]

name_mapping = dict(zip(feature_names, safe_names))
lgb_cat_columns = [name_mapping[column] for column in cat_columns]

X_train_native.columns = safe_names
X_valid_native.columns = safe_names


joblib.dump(
    {
        "original_columns": feature_names.tolist(),
        "numeric_columns": numeric_columns,
        "categorical_columns": cat_columns,
        "category_levels": category_levels,
        "model_columns": safe_names,
        "categorical_model_columns": lgb_cat_columns
    },
    ARTIFACT_DIR / "adult_lgb_input_info.joblib"
)

assert X_train_native.columns.equals(X_valid_native.columns)
assert len(X_train_native) == len(y_train)
assert len(X_valid_native) == len(y_valid)

Categorical features:
['categorical__workclass', 'categorical__marital.status', 'categorical__occupation', 'categorical__relationship', 'categorical__race', 'categorical__sex', 'categorical__native.country']


In [21]:
def lgb_objective(trial):
    model = LGBMClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        min_child_samples=trial.suggest_int(
            "min_child_samples", 20, 150
        ),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda", 0.1, 20, log=True
        ),
        objective="binary",
        random_state=SEED,
        n_jobs=THREADS,
        verbosity=-1
    )

    model.fit(
        X_train_native,
        y_train,
        categorical_feature=lgb_cat_columns
    )

    probability = model.predict_proba(X_valid_native)[:, 1]

    return roc_auc_score(y_valid, probability)


database_path = (STUDY_DIR / "adult_optuna.db").resolve()

lgb_study = optuna.create_study(
    study_name="adult_lgb_native_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

start_time = time.perf_counter()

lgb_study.optimize(
    lgb_objective,
    timeout=BUDGET_SECONDS,
    n_jobs=1,
    gc_after_trial=True
)

lgb_tuning_seconds = time.perf_counter() - start_time

lgb_study.trials_dataframe().to_csv(
    RESULT_DIR / "lgb_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", lgb_study.best_value)
print("Best parameters:", lgb_study.best_params)
print("Tuning seconds:", round(lgb_tuning_seconds, 2))

[I 2026-10-16 16:54:49,263] A new study created in RDB with name: adult_lgb_native_v1
[I 2026-10-16 16:54:51,044] Trial 0 finished with value: 0.918302739449356 and parameters: {'n_estimators': 500, 'learning_rate': 0.13125830316209655, 'num_leaves': 97, 'min_child_samples': 98, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 9.842315738502599}. Best is trial 0 with value: 0.918302739449356.
[I 2026-10-16 16:54:52,569] Trial 1 finished with value: 0.9284939684375774 and parameters: {'n_estimators': 700, 'learning_rate': 0.06803900745073706, 'num_leaves': 17, 'min_child_samples': 147, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.26425260575499165}. Best is trial 1 with value: 0.9284939684375774.
[I 2026-10-16 16:54:54,107] Trial 2 finished with value: 0.9271988349995869 and parameters: {'n_estimators': 400, 'learning_rate': 0.04141536

Best validation ROC-AUC: 0.9313848953886548
Best parameters: {'n_estimators': 700, 'learning_rate': 0.016087471410268617, 'num_leaves': 32, 'min_child_samples': 21, 'subsample': 0.9927889184287769, 'colsample_bytree': 0.6055069800913692, 'reg_alpha': 0.0007092262057066827, 'reg_lambda': 0.1039944905746209}
Tuning seconds: 600.25


In [22]:
lgb_tuned = LGBMClassifier(
    **lgb_study.best_params,
    subsample_freq=1,
    objective="binary",
    random_state=SEED,
    n_jobs=THREADS,
    verbosity=-1
)

start_time = time.perf_counter()

lgb_tuned.fit(
    X_train_native,
    y_train,
    categorical_feature=lgb_cat_columns
)

lgb_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = lgb_tuned.predict_proba(
    X_valid_native
)[:, 1]

lgb_prediction_seconds = time.perf_counter() - start_time

lgb_results = {
    "Model": "LightGBM",
    "Representation": "Native",
    "Version": "Tuned",
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, valid_probability
    ),
    "Validation_AP": average_precision_score(
        y_valid, valid_probability
    ),
    "Validation_Log_Loss": log_loss(
        y_valid, valid_probability
    ),
    "Fit_Seconds": lgb_fit_seconds,
    "Prediction_Seconds": lgb_prediction_seconds,
    "Tuning_Seconds": lgb_tuning_seconds
}

display(pd.DataFrame([lgb_results]).round(4))

joblib.dump(
    lgb_tuned,
    ARTIFACT_DIR / "adult_lgb_tuned.joblib"
)

pd.DataFrame([lgb_results]).to_csv(
    RESULT_DIR / "lgb_tuned_validation.csv",
    index=False
)

pd.Series(lgb_study.best_params, name="Value").to_csv(
    RESULT_DIR / "lgb_best_parameters.csv",
    index_label="Parameter"
)

print("Tuned LightGBM saved.")

,Model,Representation,Version,Validation_ROC_AUC,Validation_AP,Validation_Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,LightGBM,Native,Tuned,0.9314,0.8327,0.2721,1.5107,0.2117,600.2532


Tuned LightGBM saved.


## CATBOOST

In [23]:
from catboost import CatBoostClassifier

native_preprocessor = joblib.load(
    ARTIFACT_DIR / "adult_native_preprocessor.joblib"
)

feature_names = native_preprocessor.get_feature_names_out()

X_train_cat = pd.DataFrame(
    native_preprocessor.transform(X_train_raw),
    columns=feature_names,
    index=X_train_raw.index
)

X_valid_cat = pd.DataFrame(
    native_preprocessor.transform(X_valid_raw),
    columns=feature_names,
    index=X_valid_raw.index
)

numeric_columns = [
    column for column in feature_names
    if column.startswith("numeric__")
]

cat_columns = [
    column for column in feature_names
    if column.startswith("categorical__")
]

print("Categorical features:")
print(cat_columns)

# Restore numeric types; keep categorical values as strings.
for data in [X_train_cat, X_valid_cat]:
    data[numeric_columns] = data[numeric_columns].astype("float32")

    for column in cat_columns:
        data[column] = data[column].astype(str)

# Apply the same unseen-category policy as LightGBM.
category_levels = {}

for column in cat_columns:
    levels = sorted(X_train_cat[column].unique().tolist())

    if "Missing" not in levels:
        levels.append("Missing")

    category_levels[column] = levels

    X_valid_cat[column] = X_valid_cat[column].where(
        X_valid_cat[column].isin(levels),
        "Missing"
    )

assert X_train_cat.columns.equals(X_valid_cat.columns)
assert len(X_train_cat) == len(y_train)
assert len(X_valid_cat) == len(y_valid)

joblib.dump(
    {
        "original_columns": feature_names.tolist(),
        "numeric_columns": numeric_columns,
        "categorical_columns": cat_columns,
        "category_levels": category_levels,
        "model_columns": feature_names.tolist()
    },
    ARTIFACT_DIR / "adult_cat_input_info.joblib"
)


def cat_objective(trial):
    model = CatBoostClassifier(
        iterations=trial.suggest_int(
            "iterations", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        depth=trial.suggest_int("depth", 4, 8),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg", 1, 30, log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength", 0.01, 10, log=True
        ),
        bootstrap_type="Bayesian",
        bagging_temperature=trial.suggest_float(
            "bagging_temperature", 0, 5
        ),
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        thread_count=THREADS,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train_cat,
        y_train,
        cat_features=cat_columns
    )

    probability = model.predict_proba(X_valid_cat)[:, 1]

    return roc_auc_score(y_valid, probability)


database_path = (STUDY_DIR / "adult_optuna.db").resolve()

cat_study = optuna.create_study(
    study_name="adult_cat_native_10min_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

start_time = time.perf_counter()

cat_study.optimize(
    cat_objective,
    timeout=10 * 60,
    n_jobs=1,
    gc_after_trial=True
)

cat_tuning_seconds = time.perf_counter() - start_time

cat_study.trials_dataframe().to_csv(
    RESULT_DIR / "cat_10min_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", cat_study.best_value)
print("Best parameters:", cat_study.best_params)
print("Tuning seconds:", round(cat_tuning_seconds, 2))

[I 2026-10-16 17:04:51,832] A new study created in RDB with name: adult_cat_native_10min_v1


Categorical features:
['categorical__workclass', 'categorical__marital.status', 'categorical__occupation', 'categorical__relationship', 'categorical__race', 'categorical__sex', 'categorical__native.country']


[I 2026-10-16 17:05:35,067] Trial 0 finished with value: 0.9244832548403976 and parameters: {'iterations': 500, 'learning_rate': 0.13125830316209655, 'depth': 7, 'l2_leaf_reg': 7.6611007077713635, 'random_strength': 0.02938027938703535, 'bagging_temperature': 0.7799726016810132}. Best is trial 0 with value: 0.9244832548403976.
[I 2026-10-16 17:05:52,556] Trial 1 finished with value: 0.9140615962984383 and parameters: {'iterations': 200, 'learning_rate': 0.10440040750544663, 'depth': 7, 'l2_leaf_reg': 11.114989443094977, 'random_strength': 0.011527987128232402, 'bagging_temperature': 4.8495492608099715}. Best is trial 0 with value: 0.9244832548403976.
[I 2026-10-16 17:06:35,872] Trial 2 finished with value: 0.9176642384349152 and parameters: {'iterations': 900, 'learning_rate': 0.01777174904859463, 'depth': 4, 'l2_leaf_reg': 1.8659959624904914, 'random_strength': 0.08179499475211674, 'bagging_temperature': 2.6237821581611893}. Best is trial 0 with value: 0.9244832548403976.
[I 2026-10-1

Best validation ROC-AUC: 0.9296791658633765
Best parameters: {'iterations': 600, 'learning_rate': 0.09653799369770961, 'depth': 5, 'l2_leaf_reg': 4.5237872584641385, 'random_strength': 0.8875048288951415, 'bagging_temperature': 0.13955646578991487}
Tuning seconds: 624.67


In [24]:
cat_tuned = CatBoostClassifier(
    **cat_study.best_params,
    bootstrap_type="Bayesian",
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    thread_count=THREADS,
    verbose=False,
    allow_writing_files=False
)

start_time = time.perf_counter()

cat_tuned.fit(
    X_train_cat,
    y_train,
    cat_features=cat_columns
)

cat_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = cat_tuned.predict_proba(
    X_valid_cat,
    thread_count=THREADS
)[:, 1]

cat_prediction_seconds = time.perf_counter() - start_time

cat_results = {
    "Model": "CatBoost",
    "Representation": "Native",
    "Version": "Tuned",
    "Study_Name": cat_study.study_name,
    "Budget_Seconds": 600,
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, valid_probability
    ),
    "Validation_AP": average_precision_score(
        y_valid, valid_probability
    ),
    "Validation_Log_Loss": log_loss(
        y_valid, valid_probability
    ),
    "Fit_Seconds": cat_fit_seconds,
    "Prediction_Seconds": cat_prediction_seconds,
    "Tuning_Seconds": cat_tuning_seconds
}

display(pd.DataFrame([cat_results]).round(4))

joblib.dump(
    cat_tuned,
    ARTIFACT_DIR / "adult_cat_tuned.joblib"
)

pd.DataFrame([cat_results]).to_csv(
    RESULT_DIR / "cat_tuned_validation.csv",
    index=False
)

pd.Series(cat_study.best_params, name="Value").to_csv(
    RESULT_DIR / "cat_best_parameters.csv",
    index_label="Parameter"
)

print("Tuned CatBoost saved.")

,Model,Representation,Version,Study_Name,Budget_Seconds,Validation_ROC_AUC,Validation_AP,Validation_Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,CatBoost,Native,Tuned,adult_cat_native_10min_v1,600,0.9297,0.8282,0.2759,36.1999,0.0294,624.6658


Tuned CatBoost saved.
